Notebook de pruebas:
Tiene todas las funcionalidades del standalone (main_despliegue_standalone)

Nota: Si este notebook da errores de importación asociados a Cartopy (CRSS); se debe reiniciar el kernel y volver a ejecutar todo

In [ ]:
####################### N O  T O C A R ############################################
import os
import sys
import pandas as pd

root_path = os.path.abspath(os.path.join(os.path.dirname(os.getcwd()), ".."))
sys.path.append(root_path)
from pathlib import Path
from procesado_datos.config_modulo.ProcesadoConfig import ProcesadoConfig

# Importar configuraciones del modulo
from procesado_datos.config_modulo.config_procesado import config_modulo_procesado
# Importar configuraciones del submodulo de despliegue
from procesado_datos.despliegue.configs.configuracion_despliegue import config_despliegue

# Importar servicios necesarios
from procesado_datos.despliegue.services.crear_datos_lab import crear_datos_lab
from procesado_datos.config_modulo.ProcesadoConfig import ProcesadoConfig
from procesado_datos.despliegue.services.guardar_datos_lab_csv import guardar_datos_lab_csv
from procesado_datos.services.Graficado.graficar_mapa_de_despliegue import graficar_mapa_de_despliegue
from procesado_datos.services.Graficado.graficar_series_laboratorio_y_guardar import graficar_series_laboratorio_y_guardar
from procesado_datos.services.Utils.utilidades import guardar_diccionario_como_pickle
from procesado_datos.services.Graficado.graficar_mapa_prueba_lab import graficar_mapa_prueba_lab
from procesado_datos.services.Utils.utilidades import *
from procesado_datos.services.Utils.excel_a_png import *

##################################################################################

In [ ]:
# Crear la instancia del manager de configuraciones
config = ProcesadoConfig.from_sources(
    config_modulo_procesado,
    config_despliegue,
)    

In [ ]:
# Pasos:
# 1. Crear y guardar (en csv) los datos de las pruebas de laboratorio para cada sonda
datos = crear_datos_lab(config)

In [ ]:
# Crear ruta a la carpeta de guardado de datos de laboratorio
ruta_a_carpeta = config.carpeta_de_guardado_de_datos_lab 
fecha_del_estudio = config.convertir_a_pd_datetime("fecha_del_estudio", formato="%Y-%m-%d")
carpeta_del_estudio = f"{fecha_del_estudio.year:04d}{fecha_del_estudio.month:02d}"
ruta_a_la_carpeta_de_guardado = os.path.join(ruta_a_carpeta, carpeta_del_estudio, "pruebas_lab")

In [ ]:
# 2. Guardar datos datos en csv
guardar_datos_lab_csv(datos, ruta_a_la_carpeta_de_guardado)

In [ ]:
# 3. Guardar los datos procesados    
guardar_diccionario_como_pickle(
    datos, 
    ruta = ruta_a_la_carpeta_de_guardado, 
    nombre_archivo = config.nombre_del_archivo_de_datos_procesados
)


In [ ]:
# 4. Grafica la serie de tiempo de voltaje de cada sonda durante las pruebas de laboratorio
graficar_series_laboratorio_y_guardar(
    datos, 
    mostrar_figura = False, 
    ruta_a_la_carpeta_de_guardado = ruta_a_la_carpeta_de_guardado,
    config = config
)

In [ ]:
# 7. Guarda las pruebas de laboratorio en un archivo png
archivos_csv = sorted(
    Path(ruta_a_la_carpeta_de_guardado).glob("prueba_en_tierra_*_TOTAL.csv")
)

if archivos_csv:
    csv_a_png(
        [str(p) for p in archivos_csv],
        carpeta_salida=ruta_a_la_carpeta_de_guardado,
        max_filas=20,
    )
    print(f"Se generaron {len(archivos_csv)} imágenes PNG de pruebas de transmisión")
else:
    print("No se encontraron CSV para convertir")

In [ ]:
# # 5. Grafica el mapa de ubicación de las sondas durante las pruebas de laboratorio
graficar_mapa_prueba_lab(
    datos, 
    mostrar_figura=False,
    ruta_a_la_carpeta_de_guardado = ruta_a_la_carpeta_de_guardado,
    config = config
)

In [ ]:
# # 6. Grafica el mapa de despliegue
graficar_mapa_de_despliegue(
    mostrar_figura = False,
    ruta_a_la_carpeta_de_guardado = ruta_a_la_carpeta_de_guardado,
    config = config
)